# Toponym Recognition

## Goal

Extract candidate place names from text with public NER tools.

## What you will do

- Learn what each NER tool is useful for.
- Install a tool only when you want to run that section.
- Run small examples with spaCy, Stanza, Flair, and Transformers.
- Compare outputs and save mentions to `outputs/results/ner_mentions.csv`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import subprocess
import sys
import pandas as pd
from src.config import RESULTS_DIR
from src.data_utils import load_sample_texts, save_dataframe
from src.ner_utils import extract_locations_spacy, extract_locations_stanza, extract_locations_flair, extract_locations_transformers, normalize_ner_results, combine_and_deduplicate_mentions, LOCATION_LABELS

## Step 1: NER labels

NER tools use different labels. This project treats `LOC`, `LOCATION`, `GPE`, `FAC`, `FACILITY`, and `NEL` as location-like labels.

`GPE` usually means countries, cities, or states. `LOC` usually means natural or general locations. `FAC` can mean buildings, airports, bridges, or other facilities.

## Step 2: Load example texts

In [ ]:
texts = load_sample_texts()
all_mentions = []
texts

## Step 3: spaCy

spaCy is the recommended first tool for the workshop: it is lightweight, fast, and easy to explain. Run the install cell once if spaCy or the small models are missing.

In [ ]:
RUN_SPACY_INSTALL = False

if RUN_SPACY_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "de_core_news_sm"])
else:
    print("Skipped spaCy install. Set RUN_SPACY_INSTALL = True to install spaCy and the small English/German models.")

In [ ]:
rows = []
for _, row in texts.iterrows():
    results = extract_locations_spacy(row["text"], model_name="en_core_web_sm")
    rows.append(normalize_ner_results(results, source_text_id=row["text_id"]))

spacy_mentions = combine_and_deduplicate_mentions(rows)
all_mentions.append(spacy_mentions)
spacy_mentions

In [ ]:
german_text = texts.loc[texts["text_id"] == "sample_3", "text"].iloc[0]
german_spacy_mentions = normalize_ner_results(
    extract_locations_spacy(german_text, model_name="de_core_news_sm"),
    source_text_id="sample_3_de_spacy",
)
german_spacy_mentions

## Step 4: Stanza

Stanza is useful when you want a Stanford NLP pipeline and broad language coverage. Its language models are downloaded separately.

In [ ]:
RUN_STANZA_INSTALL = False

if RUN_STANZA_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "stanza"])
    import stanza
    stanza.download("en")
else:
    print("Skipped Stanza install. Set RUN_STANZA_INSTALL = True to install Stanza and download the English model.")

In [ ]:
stanza_results = extract_locations_stanza("The earthquake affected Izmir and nearby villages.", lang="en")
stanza_mentions = normalize_ner_results(stanza_results, source_text_id="stanza_demo")
all_mentions.append(stanza_mentions)
stanza_mentions

## Step 5: Flair

Flair provides strong sequence labeling models. It is heavier than spaCy, so use it when you want to compare model behavior.

In [ ]:
RUN_FLAIR_INSTALL = False

if RUN_FLAIR_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flair"])
else:
    print("Skipped Flair install. Set RUN_FLAIR_INSTALL = True to install Flair.")

In [ ]:
flair_results = extract_locations_flair("Paris and Berlin are often mentioned in European news.")
flair_mentions = normalize_ner_results(flair_results, source_text_id="flair_demo")
all_mentions.append(flair_mentions)
flair_mentions

## Step 6: Hugging Face Transformers

Transformers are useful for multilingual or domain-specific NER models. They can be large, so install them only if this section is relevant to your task.

In [ ]:
RUN_TRANSFORMERS_INSTALL = False

if RUN_TRANSFORMERS_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "torch"])
else:
    print("Skipped Transformers install. Set RUN_TRANSFORMERS_INSTALL = True to install Transformers and Torch.")

In [ ]:
transformer_results = extract_locations_transformers(
    "Nach dem Hochwasser wurden Schäden in Bayern und Passau gemeldet."
)
transformer_mentions = normalize_ner_results(transformer_results, source_text_id="transformer_demo")
all_mentions.append(transformer_mentions)
transformer_mentions

## Step 7: Compare and deduplicate mentions

In [ ]:
combined = combine_and_deduplicate_mentions(all_mentions)
combined

## Step 8: Save mentions

NER only extracts candidate place names. It does not resolve them to coordinates.

In [ ]:
out = save_dataframe(combined, RESULTS_DIR / 'ner_mentions.csv')
print('Saved:', out)

## Exercise

Add your own text and run it with at least two tools. Compare which place names are found and which labels they receive.

In [ ]:
my_text = "I travelled from Munich to Zurich and then to Vienna."
my_spacy = normalize_ner_results(extract_locations_spacy(my_text), source_text_id="my_text_spacy")
my_stanza = normalize_ner_results(extract_locations_stanza(my_text), source_text_id="my_text_stanza")
combine_and_deduplicate_mentions([my_spacy, my_stanza])

## Common issues

- If a package is missing, run the install cell in that tool's section.
- If spaCy is installed but the model is missing, rerun the spaCy install cell.
- Stanza, Flair, and Transformers may download larger models on first use.
- Some `FAC` entities are useful locations; others are not, depending on the application.